In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score, accuracy_score, mean_absolute_error

In [ ]:
import torch
import torch.nn as nn 
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim 
from torch.optim.lr_scheduler import ReduceLROnPlateau


In [ ]:
import random
import copy
from collections import defaultdict 

## Baseline Data

In [ ]:
df_baseline = pd.read_csv('Baseline_FeatureSelection.csv')

In [ ]:
df_baseline

In [ ]:
df_baseline['user_order'] = df_baseline.index + 1

In [ ]:
df_baseline.columns

In [ ]:
baseline_feature_list = ['VCI', 'PRI', 'WMI', 'IQ', 'alcohol_freq', '순수알코올량',
       'DMQ1_F2', 'DMQ1_F3', 'DMQ1_F4', 'BIS1_t', 'SOCRATES1_F1',
        'delta-T7', 'delta-CP6', 'delta-P2', 'theta-C4',
       'theta-CPz', 'theta-TP8', 'theta-Oz', 'alpha-Fp1', 'alpha-F7',
       'beta-F5', 'beta-F1', 'beta-F6', 'beta-FCz', 'beta-T8', 'beta-CP5',
       'beta-Oz', 'gamma-F3', 'gamma-FC2', 'gamma-T7', 'gamma-TP7', 'gamma-P7',
       'gamma-O2', '자율신경 활성도', '자율신경 균형도', '심장 안정도', 'Tp', 'Apen', 'Srd',
       'Vlf(ln)', '심장 안정도 상태 - Encoded']

baseline_user_col = 'user_order'

In [ ]:
len(baseline_feature_list)

In [ ]:
df_baseline = df_baseline.drop(index = 34)

In [ ]:
df_baseline.user_order.unique()

In [ ]:
S = df_baseline[baseline_feature_list]

In [ ]:
S_matrix = df_baseline[baseline_feature_list].to_numpy(dtype = np.float32)
S_user = df_baseline[baseline_user_col].to_numpy()

baseline_dict  = {u: S_matrix[i] for i, u in enumerate(S_user)}

## App Data

In [ ]:
df_app = pd.read_csv('App_FeatureSelection.csv')

In [ ]:
df_app = df_app.sort_values(by = ['user_id', 'date'])
df_app['date'] = pd.to_datetime(df_app['date'])
df_app['day_order'] = (df_app.date - df_app.groupby('user_order')['date'].transform('min')).dt.days + 1
df_app['date_diff'] = df_app.groupby('user_order')['date'].diff().dt.days

df_app = df_app.drop(index = [1124, 1125, 1126])
df_app = df_app[df_app['user_order'] != 35]
df_app = df_app.reset_index(drop = True)


In [ ]:
df_app = df_app.groupby('user_order').head(27).reset_index(drop = True)

In [ ]:
df_app

In [ ]:
feature_list = ['phq1', 'phq2', 'meal_morning', 'meal_lunch', 'meal_dinner',
       'meal_night', 'sleep_satisfy', 'sleep_disturb', 'amount_beer_glass',
       'amount_soju_glass', 'sleep_minute_adjusted']

target_variable = 'tmrw_alcohol_desire'

In [ ]:
X = df_app[feature_list]
y = df_app[target_variable]

In [ ]:
df_app.user_order.unique()

In [ ]:
X_train, y_train, user_train, S_train = [], [], [], []
X_val, y_val, user_val, S_val = [], [], [], []
X_test, y_test, user_test, S_test = [], [], [], []

In [ ]:
window = 4
val_days = 3
test_days = 5

In [ ]:
for user, user_data in df_app.groupby('user_order'):
    
    
    X_user = user_data[feature_list].to_numpy(dtype = np.float32)
    y_user = user_data[target_variable].to_numpy()
    
    T = len(user_data)
    
    # Index 기준: Starts from 0
    test_start_day = T - test_days
    val_start_day = T - (val_days + test_days)
    train_end_day = val_start_day - 1 
    
    S_user = baseline_dict[user]
    
    
    for start in range(0, T - window + 1):
        
        target_idx = start + window - 1
        
        X_window = X_user[start : start + window]
        y_window = y_user[target_idx] 
        y_window = int(y_window) - 1 # [1,2,3,4,5] -> [0,1,2,3,4]
        
        
        # 1. Check Within Window
        day_list = user_data['day_order'].iloc[start : start + window]
        day_array = np.array(day_list)
        flag = (np.diff(day_array) == 1).all()
        
        if flag != True:
            continue
        # ==================================================================
        
        # 2. Check Next Day Target
        if target_idx + 1 < T:
            last_input_day = user_data['day_order'].iloc[target_idx]
            next_day = user_data['day_order'].iloc[target_idx + 1]
        
            if next_day - last_input_day != 1:
                continue
        # ==================================================================
        
        # Train, Val, Test Split
        if target_idx <= train_end_day:
            X_train.append(X_window)
            y_train.append(y_window)
            user_train.append(user)
            S_train.append(S_user)

            
        elif val_start_day <= target_idx < test_start_day:
            X_val.append(X_window)
            y_val.append(y_window)
            user_val.append(user)
            S_val.append(S_user)

            
        elif test_start_day <= target_idx:
            X_test.append(X_window)
            y_test.append(y_window)
            user_test.append(user)
            S_test.append(S_user)

        
        
        

In [ ]:
X_train, y_train, user_train, S_train = np.array(X_train), np.array(y_train), np.array(user_train), np.array(S_train)
X_val, y_val, user_val, S_val = np.array(X_val), np.array(y_val), np.array(user_val), np.array(S_val)
X_test, y_test, user_test, S_test = np.array(X_test), np.array(y_test), np.array(user_test), np.array(S_test)

In [ ]:
X_train.shape

### numpy() -> pytorch Tensor

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
y_train_tensor = torch.from_numpy(y_train).long()

X_val_tensor = torch.from_numpy(X_val)
y_val_tensor = torch.from_numpy(y_val).long()

X_test_tensor = torch.from_numpy(X_test)
y_test_tensor = torch.from_numpy(y_test).long()

user_train_tensor = torch.from_numpy(user_train)
user_val_tensor = torch.from_numpy(user_val)
user_test_tensor = torch.from_numpy(user_test)

S_train_tensor = torch.from_numpy(S_train)
S_val_tensor = torch.from_numpy(S_val)
S_test_tensor = torch.from_numpy(S_test)

In [ ]:
train_dataset = TensorDataset(X_train_tensor, S_train_tensor, y_train_tensor, user_train_tensor)
val_dataset = TensorDataset(X_val_tensor, S_val_tensor, y_val_tensor, user_val_tensor)
test_dataset = TensorDataset(X_test_tensor, S_test_tensor,y_test_tensor, user_test_tensor)

## DataLoader

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle= True, drop_last= False)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size = 256, shuffle=False, drop_last=False)

### Baseline Data -> $h_0$

In [ ]:
class BaselinetoH0(nn.Module):
    
    def __init__(self, static_size, hidden_size, p_drop = 0.5):
        super().__init__()
        
        self.BaselineModel = nn.Sequential(
            nn.Linear(static_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden_size, hidden_size),
        )
    
    def forward(self, s):    
        # s: Baseline Data
        h0 = torch.tanh(self.BaselineModel(s))
        return h0 
    

### Vanilla RNN Model

In [ ]:
class TinyVanillaRNN(nn.Module):
    
    def __init__(self, input_size, hidden_size, static_size , task = 'cls', num_classes = 5, p_drop = 0.5):
        super().__init__()
        
        self.task = task 
        self.rnn = nn.RNN(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = 1,
            batch_first = True,
            nonlinearity = 'tanh'
        )
        
        self.h0_encoder = BaselinetoH0(static_size, hidden_size, p_drop = p_drop)
        self.head = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x, s):
        # x : (Batch, Window, Feature)
        # s : (Batch, static_size)
        
        h0 = self.h0_encoder(s) # (Batch, hidden_size)
        h0 = h0.unsqueeze(0) # (1, Batch, hidden_size)
        
        out, h_n = self.rnn(x, h0) # h_n: (1, Batch, hidden_size)
        h_last = h_n[-1] # (Batch, hidden_size)
        y_hat = self.head(h_last) # (Batch, num_class) s
        
        return y_hat 

## Run 100 Times

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [ ]:
hidden_size_list = [8,16,32]

In [ ]:
input_size = X_train_tensor.shape[2]
static_size = S_train.shape[1]
hidden_size = hidden_size_list[1]
EPOCHS = 70


In [ ]:
def train_one_run(run_id, seed):
    
    # Set Seed
    set_seed(seed)

    # Define a New Model each Time
    model = TinyVanillaRNN(
    input_size=input_size,
    hidden_size=hidden_size,
    static_size=static_size,
    num_classes=5,
    p_drop=0.2
)
    
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    history = []
    
    best_val_loss = float('inf')
    best_epoch = None
    best_state = None
    

    # 2) train loop
    for epoch in range(EPOCHS):
        
        model.train() # Training Mode
        
        total_loss = 0.0
        train_correct = 0
        train_total = 0
        
        
        for X_batch, S_batch, y_batch, user_batch in train_loader:


            optimizer.zero_grad() # Initialize Weights
            logits = model(X_batch, S_batch)
            loss = criterion(logits, y_batch) # Calculate Loss 
            loss.backward() # Back Propagation -> Calculate Derivatives
            optimizer.step() # Update Weights
            
            ttotal_loss = total_loss + loss.item() * X_batch.size(0)
            pred = logits.argmax(dim = 1)
            train_correct = train_correct + (pred == y_batch).sum().item()
            train_total = train_total + y_batch.size(0)
        
        train_metrics = eval_metrics_on_loader(model, train_loader, criterion=criterion)
        val_metrics   = eval_metrics_on_loader(model, val_loader,   criterion=criterion)

    # Append to the History List
        history.append({
                "run": run_id,
                "seed": seed,
                
                "epoch": epoch + 1,
                
                "train_loss": train_metrics["loss"],
                "train_acc": train_metrics["acc"],
                "train_macro_f1": train_metrics["macro_f1"],
                "train_weighted_f1": train_metrics["weighted_f1"],
                "train_mae": train_metrics["mae"],
                
                "val_loss": val_metrics["loss"],
                "val_acc": val_metrics["acc"],
                "val_macro_f1": val_metrics["macro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_mae": val_metrics["mae"],
            })
        
        # Update Best Checkpoint
        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
        
    # Load Weights of the best checkpoint
    model.load_state_dict(best_state)  

    # After Total Model Training => Final Evaluation
    final_train_metrics = eval_metrics_on_loader(model, train_loader, criterion=criterion)
    final_val_metrics   = eval_metrics_on_loader(model, val_loader,   criterion=criterion)
    final_test_metrics = eval_metrics_on_loader(model, test_loader, criterion=criterion)
        

    summary = {
        "run": run_id,
        "seed": seed,
        
        # Add Best Epoch, Best Val_loss
        'best_epoch' : best_epoch,
        'best_val_loss': best_val_loss,

        # Train (final model)
        "train_loss": final_train_metrics["loss"],
        "train_acc": final_train_metrics["acc"],
        "train_macro_f1": final_train_metrics["macro_f1"],
        "train_weighted_f1": final_train_metrics["weighted_f1"],
        "train_macro_precision": final_train_metrics["macro_precision"],
        "train_macro_recall": final_train_metrics["macro_recall"],
        "train_mae": final_train_metrics["mae"],

        # Val (final model)
        "val_loss": final_val_metrics["loss"],
        "val_acc": final_val_metrics["acc"],
        "val_macro_f1": final_val_metrics["macro_f1"],
        "val_weighted_f1": final_val_metrics["weighted_f1"],
        "val_macro_precision": final_val_metrics["macro_precision"],
        "val_macro_recall": final_val_metrics["macro_recall"],
        "val_mae": final_val_metrics["mae"],

        # Test (final model)
        "test_loss": final_test_metrics["loss"],
        "test_acc": final_test_metrics["acc"],
        "test_macro_f1": final_test_metrics["macro_f1"],
        "test_weighted_f1": final_test_metrics["weighted_f1"],
        "test_macro_precision": final_test_metrics["macro_precision"],
        "test_macro_recall": final_test_metrics["macro_recall"],
        "test_mae": final_test_metrics["mae"],
        
        
    }
    
    # # Participant Wise Accuracy
    # train_user_acc = final_train_metrics['user_acc']
    # val_user_acc = final_val_metrics['user_acc']
    # test_user_acc = final_test_metrics['user_acc']
  
    
    #  # Get Predicted Alcohol Desire Values for each day
    # df_train_pred = predict_on_loader(model, train_loader)
    # df_val_pred = predict_on_loader(model, val_loader)
    # df_test_pred = predict_on_loader(model, test_loader)
        
    return summary, history

In [ ]:
@torch.no_grad()
def eval_metrics_on_loader(model, loader, criterion=None):
    model.eval()

    total = 0
    correct = 0
    loss_sum = 0.0

    all_pred = []
    all_true = []

    for X_batch, S_batch, y_batch, user_batch in loader:
        logits = model(X_batch, S_batch)

        loss = criterion(logits, y_batch)
        loss_sum += loss.item() * X_batch.size(0)

        pred = logits.argmax(dim=1)

        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

        # PyTorch Tensor -> Numpy
        all_pred.append(pred.detach().numpy())
        all_true.append(y_batch.detach().numpy())

    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)

    metrics = {
        "loss": (loss_sum / total),
        "acc": correct / total,
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "mae": mean_absolute_error(y_true, y_pred),
    }
    return metrics

In [ ]:
N_RUNS = 100
base_seed = 2026
seeds = [base_seed + i for i in range(N_RUNS)]

In [ ]:
all_summaries = []
all_histories = []

In [ ]:
for id, seed in enumerate(seeds):
    summary, history = train_one_run(run_id=id, seed=seed)
    all_summaries.append(summary)
    all_histories.extend(history)

df_runs = pd.DataFrame(all_summaries)     
df_hist = pd.DataFrame(all_histories) 

In [ ]:
df_runs

In [ ]:
cols = ["train_acc", "train_macro_f1", "train_weighted_f1", "train_mae",
        "val_acc", "val_macro_f1", "val_weighted_f1", "val_mae",
        "test_acc", "test_macro_f1", "test_weighted_f1", "test_mae",]

In [ ]:
df_runs[cols].agg(["mean","std"]).T.round(4)

In [ ]:
1500 / 6

In [ ]:
250 * 6